In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import datania

# Load data (smaller sample to show quality variation)
survey_data = datania.generate_household_survey(n_households=600, seed=42)
df = pd.read_csv(survey_data)

print("Household Survey Data Loaded")
print(f"Sample size: {len(df)}")
print(f"Provinces: {df['province'].unique()}")

income = df["monthly_income"].to_numpy()

# Task 1: Calculate CV for the full sample
mean_income = np.mean(income)
std_income = np.std(income)

# YOUR CODE HERE: CV = (std / mean) * 100
cv = ___

print(f"\n=== Full Sample Quality Metrics ===")
print(f"Mean income: {mean_income:,.2f} DKW")
print(f"Std deviation: {std_income:,.2f} DKW")
print(f"CV: {cv:.1f}%")

# Task 2: Calculate RSE for the full sample
se_income = stats.sem(income)

# YOUR CODE HERE: RSE = (SE / mean) * 100
rse = ___

print(f"Standard error: {se_income:,.2f} DKW")
print(f"RSE: {rse:.2f}%")

# Task 3 & 4: Quality assessment by province
print(f"\n=== Provincial Quality Assessment ===")
print("-" * 60)
print(f"{'Province':<12} {'N':>6} {'Mean':>12} {'SE':>10} {'RSE':>8} {'Flag'}")
print("-" * 60)

results = []
for province in sorted(df["province"].unique()):
    prov_data = df[df["province"] == province]["monthly_income"].to_numpy()
    n = len(prov_data)
    prov_mean = np.mean(prov_data)
    prov_se = stats.sem(prov_data)

    # YOUR CODE HERE: Calculate RSE for this province
    prov_rse = ___

    # Determine flag
    if prov_rse > 25:
        flag = "⛔ SUPPRESS"
    elif prov_rse > 15:
        flag = "⚠️ Caution"
    else:
        flag = "✓ OK"

    results.append({
        'province': province,
        'n': n,
        'mean': prov_mean,
        'se': prov_se,
        'rse': prov_rse,
        'flag': flag
    })

    print(f"{province:<12} {n:>6} {prov_mean:>12,.0f} {prov_se:>10,.0f} {prov_rse:>7.1f}% {flag}")

print("-" * 60)

# Task 5: Summary of flagged estimates
print(f"\n=== Summary ===")
caution_count = sum(1 for r in results if '⚠️' in r['flag'])
suppress_count = sum(1 for r in results if '⛔' in r['flag'])

print(f"Estimates OK for publication: {6 - caution_count - suppress_count}")
print(f"Estimates requiring caution note: {caution_count}")
print(f"Estimates to suppress: {suppress_count}")

if suppress_count > 0:
    print("\nProvinces to suppress:")
    for r in results:
        if '⛔' in r['flag']:
            print(f"  - {r['province']} (RSE = {r['rse']:.1f}%)")